In [1]:
!pip install scikit-learn joblib pytesseract pillow

In [2]:
import os
from PIL import Image, ImageDraw

# Folders create karo
for folder in ['invoices', 'receipts', 'contracts']:
    os.makedirs(f'training_data/{folder}', exist_ok=True)

# Dummy invoice texts
invoice_texts = [
    "INVOICE #INV-001 Date: Jan 15, 2024 Total: $1,250",
    "INVOICE Number: 12345 Date: Feb 20, 2024 Amount: $500",
    "BILL TO: Acme Corp Invoice Date: March 10, 2024 Due: $750",
    "INVOICE INV-2024-001 March 5, 2024 Total $2,000",
    "Purchase Order #PO-123 Date: April 1, 2024 Amount: $3,250"
]

# Dummy receipt texts
receipt_texts = [
    "RECEIPT Walmart Date: 01/15/2024 Total: $45.67",
    "THANK YOU Receipt #123 Date: Feb 14, 2024 Amount: $23.45",
    "RECEIPT Starbucks Date: 03/20/2024 Total: $12.50",
    "RECEIPT Amazon April 5, 2024 Total $89.99",
    "RECEIPT - Target March 1, 2024 Amount $34.50"
]

# Dummy contract texts
contract_texts = [
    "CONTRACT between Party A and Party B Date: Jan 01, 2024",
    "LEASE AGREEMENT Signed: Feb 28, 2024 Effective: March 1, 2024",
    "SERVICE CONTRACT dated March 15, 2024 between Company X and Client Y",
    "EMPLOYMENT AGREEMENT April 10, 2024 effective May 1, 2024",
    "NON-DISCLOSURE AGREEMENT Date: March 25, 2024"
]

def create_dummy_image(text, filename):
    img = Image.new('RGB', (800, 200), color='white')
    draw = ImageDraw.Draw(img)
    draw.text((50, 80), text[:100], fill='black')
    img.save(filename)

# Create 5 images per class
for i, text in enumerate(invoice_texts):
    create_dummy_image(text, f'training_data/invoices/invoice_{i}.jpg')

for i, text in enumerate(receipt_texts):
    create_dummy_image(text, f'training_data/receipts/receipt_{i}.jpg')

for i, text in enumerate(contract_texts):
    create_dummy_image(text, f'training_data/contracts/contract_{i}.jpg')

print(" 15 training images created (5 per class)")

 15 training images created (5 per class)


In [3]:
import pytesseract
from PIL import Image

def load_documents(data_dir):
    documents = []
    labels = []
    
    for doc_type in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, doc_type)
        if not os.path.isdir(folder_path):
            continue
        
        for filename in os.listdir(folder_path):
            if filename.endswith('.jpg'):
                file_path = os.path.join(folder_path, filename)
                img = Image.open(file_path)
                text = pytesseract.image_to_string(img)
                documents.append(text)
                labels.append(doc_type)
    
    return documents, labels

documents, labels = load_documents('training_data')
print(f'Loaded {len(documents)} documents')
print(f'Classes: {set(labels)}')

Loaded 15 documents
Classes: {'receipts', 'invoices', 'contracts'}


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))

# Transform
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train classifier
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

# Evaluate
y_pred = classifier.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f'\n Accuracy: {accuracy:.2%}')
print('\n Classification Report:')
print(classification_report(y_test, y_pred))

Training samples: 12
Test samples: 3

 Accuracy: 100.00%

 Classification Report:
              precision    recall  f1-score   support

   contracts       1.00      1.00      1.00         1
    invoices       1.00      1.00      1.00         1
    receipts       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



In [5]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Training data
documents = [
    "INVOICE #INV-001 Date: Jan 15, 2024 Total: $1,250",
    "INVOICE Number: 12345 Date: Feb 20, 2024 Amount: $500",
    "INVOICE INV-2024-001 March 5, 2024 Total $2,000",
    "RECEIPT Walmart Date: 01/15/2024 Total: $45.67",
    "RECEIPT Starbucks Date: 03/20/2024 Total: $12.50",
    "RECEIPT Amazon April 5, 2024 Total $89.99",
    "CONTRACT AGREEMENT Date: Jan 01, 2024 between Party A and Party B",
    "LEASE CONTRACT Date: Feb 28, 2024 Landlord: John Smith",
    "SERVICE CONTRACT dated March 15, 2024 between Company X and Client Y"
]

labels = ["invoice", "invoice", "invoice", "receipt", "receipt", "receipt", "contract", "contract", "contract"]

# Train model
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
classifier = LogisticRegression()
classifier.fit(X, labels)

# Save models
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(classifier, 'classifier.pkl')

print(" Models created successfully!")
print(" vectorizer.pkl")
print(" classifier.pkl")

 Models created successfully!
 vectorizer.pkl
 classifier.pkl
